# Exploring and Streaming Earth Observation Data via the CDSE STAC API

This notebook demonstrates how to:
1. Connect to the Copernicus Data Space Ecosystem (CDSE) STAC catalogue and explore available collections
2. Define an area of interest and search for scenes matching your criteria
3. Inspect search results and understand the available assets
4. Stream asset data directly from S3 into memory for further processing

Sentinel-2 L2A is used as a worked example, but the same approach applies to any collection in the catalogue.

## Libraries

First, ensure all required packages are installed and imported.

In [0]:
%pip install --upgrade stackstac rioxarray && pip install folium pystac_client geogif s3fs stacmap shapely rasterio


In [0]:
dbutils.library.restartPython()


In [0]:
import os
from pathlib import Path
import shutil
import urllib.parse
from urllib.parse import urlparse
from collections.abc import Mapping
import subprocess
import pandas as pd
from shapely.geometry import shape
from rasterio.crs import CRS
from rasterio.warp import transform_bounds
import folium
from folium.plugins import Draw
import xarray as xr
import rioxarray
import fsspec
import geogif
import json
import pystac_client
import requests
import s3fs
import stackstac
import rasterio

Key libraries:
* `pystac_client`: Connect to and search STAC catalogues
* `s3fs` / `rioxarray`: Stream raster assets directly from S3
* `stackstac`: Build lazy multi-temporal/multi-band datacubes from STAC search results
* `folium`: Interactive map widget for defining the area of interest

## S3 Credentials

In order to access the data from CloudFerro's catalog, authentication is required. To facilitate this, you will need to create an account in the Copernicus Data Space Ecosystem (CDSE). The registration process is quick and can be completed in under 60 seconds. You can find the registration link here: https://documentation.dataspace.copernicus.eu/APIs/S3.html.

Once you have created an account, the same URL will redirect you to the section where you can generate your `S3 Credentials`. To do this, simply click the `Add Credential` button. This will allow you to define an expiration date for your credentials.

After selecting the expiration date, you will have successfully created your own S3 credentials. Please make sure to save the secret key at this point, as it will not be displayed again.

Now that you have your S3 credentials, you can use them to connect to the STAC catalog. By utilizing the `os` package, you will set environment variables to authenticate yourself as a valid user for the S3 protocol.

The only variables you need to configure are:
* `AWS_ACCESS_KEY_ID`
* `AWS_SECRET_ACCESS_KEY`.

For this we will use [Databricks Secrets](https://learn.microsoft.com/en-us/azure/databricks/security/secrets/)

In [0]:
os.environ["GDAL_HTTP_TCP_KEEPALIVE"] = "YES"
os.environ["AWS_S3_ENDPOINT"] = "eodata.dataspace.copernicus.eu"
os.environ["AWS_ACCESS_KEY_ID"] = dbutils.secrets.get(scope = "tomkdefra_cdse", key = "AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = dbutils.secrets.get(scope = "tomkdefra_cdse", key = "AWS_SECRET_ACCESS_KEY")
os.environ["AWS_HTTPS"] = "YES"
os.environ["AWS_VIRTUAL_HOSTING"] = "FALSE"
os.environ["GDAL_HTTP_UNSAFESSL"] = "YES"

## Connecting to the STAC Catalog

We connect to the Copernicus STAC catalog, available at the endpoint: https://stac.dataspace.copernicus.eu/v1.

In [0]:
URL = "https://stac.dataspace.copernicus.eu/v1"
cat = pystac_client.Client.open(URL)
cat.add_conforms_to("ITEM_SEARCH")

## Defining the Area of Interest (AOI)

We define the area as a polygon based on geographical coordinates in GeoJSON format. We are going to get these interactively using a map widget in the notebook. Running the next cell will load a map widget, with a drag to draw polygon feature enabled. Once you have drawn a polygon, left-click the 'Export' button to activate the GeoJSON download, and then right-click the same button and chose 'copy link'.

* `type`: Geometry type, in this case, a polygon.
* `coordinates`: A list of coordinates defining the area.

In [0]:
center = [52.561928, -1.464854]
zoom = 7
m = folium.Map(location=center, zoom_start=zoom)
draw = Draw(
    export=True,
    draw_options={
        'polyline': False,
        'circlemarker': False,
        'polygon': False,
        'circle': False,
        'rectangle': True,
        'marker': True,
        'edit': True
    },
)

draw.add_to(m)
display(m)

def extract_polygon_coords(polygon_export_link):
    try:
        json_string = polygon_export_link.split(',', 1)[1]
        decoded_json = urllib.parse.unquote(json_string)
        geojson = json.loads(decoded_json)

        return geojson['features'][0]['geometry']['coordinates']
    except (IndexError, KeyError, json.JSONDecodeError) as e:
        raise ValueError("Invalid link. Please activate the link first by left-clicking the 'Export' button, before copying the link.") from e


### Extracting the coordinates

Paste the link you just copied in place of the one which begins with "data:text/json" in the next cell, then run the cell. The extracted coordinates will be passed to the rest of the notebook. 

In [0]:
polygon_export_link = "data:text/json;charset=utf-8,%7B%22type%22%3A%22FeatureCollection%22%2C%22features%22%3A%5B%7B%22type%22%3A%22Feature%22%2C%22properties%22%3A%7B%7D%2C%22geometry%22%3A%7B%22type%22%3A%22Polygon%22%2C%22coordinates%22%3A%5B%5B%5B-0.201874%2C51.47442%5D%2C%5B-0.201874%2C51.524459%5D%2C%5B-0.091324%2C51.524459%5D%2C%5B-0.091324%2C51.47442%5D%2C%5B-0.201874%2C51.47442%5D%5D%5D%7D%7D%5D%7D"

# Decode the URL string
encoded_json = polygon_export_link.split(',', 1)[1]
decoded_json_str = urllib.parse.unquote(encoded_json)

# Parse the GeoJSON FeatureCollection
feature_collection = json.loads(decoded_json_str)
aoi_feature = feature_collection['features'][0]
aoi_geometry = aoi_feature['geometry']

# Create a Shapely object and extract the WGS84 bounding box (min_lon, min_lat, max_lon, max_lat)
aoi_polygon = shape(aoi_geometry)
aoi_bounds_latlon = aoi_polygon.bounds
print(f"BBOX for stackstac: {aoi_bounds_latlon}")

# Prepare the geometry for the STAC search
geom_for_search = {
    "intersects": aoi_geometry
}

## Exploring the Catalogue

Before searching, list all available collections to see what data is in the catalogue.

In [0]:
# List all available collections (catalogs) in the STAC API
collections = cat.get_collections()
collections_data = [{"id": c.id, "title": c.title, "description": c.description} for c in collections]

display(collections_data)

## Searching for Items

Search the catalogue with your chosen filters. Key parameters:
* `collections`: which dataset to search (use the `id` from the table above)
* `datetime`: time range in `YYYY-MM-DD/YYYY-MM-DD` format
* `intersects`: geometry defining the area of interest
* `query`: property filters, e.g. maximum cloud cover

In [0]:
DATE_RANGE = "2025-05-01/2025-10-30"

search_params = {
    "max_items": 100,
    "collections": "sentinel-2-l2a",  # Paste the collection id from the table above
    "datetime": DATE_RANGE,
    **geom_for_search,
    "query": {
        "eo:cloud_cover": {"lte": 10},
    },
}
items = list(cat.search(**search_params).items())
print(f"Found {len(items)} items matching your search parameters.")

## Visualise Results on a Map (optional)

In [0]:
# stacmap is causing trouble...
# stacmap.explore(
#     items,
#     prop="eo:cloud_cover",
#     thumbnails=True
# )

## Inspecting Search Results

First, summarise the items returned by the search — date, cloud cover, and scene ID.
Then inspect the available asset keys for a single item to see what can be streamed.

In [0]:
# Summary table of search results
results_df = pd.DataFrame([
    {
        "id": item.id,
        "datetime": item.properties.get("datetime", "")[:10],
        "cloud_cover": item.properties.get("eo:cloud_cover"),
    }
    for item in items
])
display(results_df)

In [0]:
# Asset keys available for the first item
item = items[0]
assets = item.assets
print(f"Item: {item.id}")
print(f"\nAvailable assets ({len(assets)}):")
for key, asset in assets.items():
    print(f"  {key:<20} {asset.media_type or '':35} {asset.href}")

## Building a Lazy Multi-Temporal Array

`stackstac` stacks all search results into a lazy `(time, band, y, x)` dask array.
No data is read from S3 until `.compute()` is called — this just builds the graph.

Choose which asset keys to include. Run the cell above to see what's available.

In [0]:
# Choose which assets (bands) to include in the datacube.
# Use the asset keys printed above.
selected_assets = ["B04_10m", "B08_10m"]

In [0]:
# gdal_env passes credentials to every GDAL read, including on distributed workers.
# Required for non-AWS S3 endpoints — without this, .compute() will fail on Spark workers
# because they inherit no AWS_* env vars from the driver.
gdal_env = stackstac.DEFAULT_GDAL_ENV.updated(always={
    "GDAL_HTTP_TCP_KEEPALIVE": "YES",
    "AWS_S3_ENDPOINT": "eodata.dataspace.copernicus.eu",
    "AWS_ACCESS_KEY_ID": os.environ["AWS_ACCESS_KEY_ID"],
    "AWS_SECRET_ACCESS_KEY": os.environ["AWS_SECRET_ACCESS_KEY"],
    "AWS_HTTPS": "YES",
    "AWS_VIRTUAL_HOSTING": "FALSE",
    "GDAL_HTTP_UNSAFESSL": "YES",
})

cube = stackstac.stack(
    items,
    assets=selected_assets,
    bounds_latlon=aoi_bounds_latlon,
    epsg=27700,  # British National Grid
    resolution=10,  # Native 10m resolution
    chunksize=4096,
    gdal_env=gdal_env,
)

print(f"Datacube shape: {cube.shape}")
print(f"Lazy datacube ready — shape: {cube.shape} (time, band, y, x)")
print("Call .compute() on a subset when you're ready to stream data.")
display(cube)

## Streaming Assets Directly from S3

The STAC item assets contain `s3://` hrefs pointing directly to the data files on CDSE's S3 storage.
We configure an `s3fs` filesystem for listing and access verification, and use GDAL's `/vsis3/`
virtual filesystem for lazy raster reads via `rioxarray`.

In [0]:
fs = s3fs.S3FileSystem(
    anon=False,
    key=os.environ["AWS_ACCESS_KEY_ID"],
    secret=os.environ["AWS_SECRET_ACCESS_KEY"],
    client_kwargs={
        "endpoint_url": "https://eodata.dataspace.copernicus.eu",
        "verify": False,
    },
)
print("S3 filesystem configured.")

### Verify S3 Access

Confirm the credentials and endpoint are working before attempting to stream.

In [0]:
href = items[0].assets["B04_10m"].href
print(f"Asset href: {href}")

info = fs.info(href)
print(f"Accessible: True")
print(f"Size:       {info['size'] / 1e6:.1f} MB")

### Stream a Band (Lazy)

Convert the `s3://` href to a GDAL `/vsis3/` path and open it as a lazy dask-backed `DataArray`.
No data is transferred yet — only the file metadata (shape, CRS, dtype) is read.

Sentinel-2 L2A files are JPEG2000 with internal 1024×1024 tiles, which GDAL uses to
serve windowed reads — so only the requested spatial region is downloaded.

In [0]:
# Convert s3:// href to GDAL /vsis3/ path
vsi_href = items[0].assets["B04_10m"].href.replace("s3://", "/vsis3/")

red = rioxarray.open_rasterio(vsi_href, chunks={"x": 1024, "y": 1024}, lock=False)

print(red)
print(f"\nCRS:   {red.rio.crs}")
print(f"Shape: {red.shape}")
print(f"dtype: {red.dtype}")

### Clip to AOI and Stream

The AOI bounds are in WGS84 (EPSG:4326); reproject them to the band's native CRS before clipping.
Calling `.compute()` triggers the actual S3 read — only the clipped window is transferred.

In [0]:
src_crs = CRS.from_epsg(4326)
dst_crs = red.rio.crs

min_lon, min_lat, max_lon, max_lat = aoi_bounds_latlon
left, bottom, right, top = transform_bounds(src_crs, dst_crs, min_lon, min_lat, max_lon, max_lat)

red_aoi = red.rio.clip_box(minx=left, miny=bottom, maxx=right, maxy=top)
print(f"Lazy clipped shape: {red_aoi.shape}")

# Stream only the clipped window from S3
red_data = red_aoi.compute()
print(f"Streamed. Value range: {int(red_data.min())} – {int(red_data.max())}")

# The DataArray is now in memory and ready for any further processing
red_data.squeeze().plot(robust=True, cmap="Greys_r")